# Simple Iceberg Backup

## Setup Spark session for multi-region

We setup the Spark session with 2 Iceberg catalogs, one per region (source and target).
In this example from "eu-west-1" (current region) to "us-east-1" (target region). Customize regions as needed

In [ ]:
%%configure -f
{   
    "conf":{
         "spark.sql.extensions":"org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
         "spark.hadoop.hive.metastore.client.factory.class":"com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory",
         "spark.sql.catalog.iceberg_catalog_region1":"org.apache.iceberg.spark.SparkCatalog",
         "spark.sql.catalog.iceberg_catalog_region1.type": "glue",
         "spark.sql.catalog.iceberg_catalog_region1.glue.region":"eu-west-1",
         "spark.sql.catalog.iceberg_catalog_region1.s3.region":"eu-west-1",
         "spark.sql.catalog.iceberg_catalog_region1.client.region":"eu-west-1",
         "spark.sql.catalog.iceberg_catalog_region2":"org.apache.iceberg.spark.SparkCatalog",
         "spark.sql.catalog.iceberg_catalog_region2.type": "glue",
         "spark.sql.catalog.iceberg_catalog_region2.glue.region":"us-east-1",
         "spark.sql.catalog.iceberg_catalog_region2.s3.region":"us-east-1",
         "spark.sql.catalog.iceberg_catalog_region2.client.region":"us-east-1"
    }
}

## Setup variables and libraries

In [ ]:
from datetime import datetime
TABLE_NAME="iceberg_table_1"
DB_NAME="mydb_replication"
S3_ROOT_REGION_1="s3://my_eu_west_1_bucket/"
S3_ROOT_REGION_2="s3://my_us_east_1_bucket/"

## Create DB and table in the current region

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS iceberg_catalog_region1.{DB_NAME} LOCATION '{S3_ROOT_REGION_1}/{DB_NAME}/'
""")

In [ ]:
spark.sql(f"""
CREATE EXTERNAL TABLE iceberg_catalog_region1.{DB_NAME}.{TABLE_NAME}(
    id INT,
    col1 INT,
    col2 INT,
    col3 INT,
    col4 INT,
    country string
)
USING ICEBERG
PARTITIONED BY (country)
LOCATION '{S3_ROOT_REGION_1}/{DB_NAME}/{TABLE_NAME}/'
""")

In [ ]:
spark.sql(f"""
    INSERT INTO iceberg_catalog_region1.{DB_NAME}.{TABLE_NAME}
    VALUES 
    (1, 100, 200, 300, 400, 'italy'),
    (2, 101, 201, 301, 401, 'france')
""")

In [ ]:
spark.sql(f"""
    INSERT INTO iceberg_catalog_region1.{DB_NAME}.{TABLE_NAME}
    VALUES 
    (3, 102, 202, 302, 402, 'italy'),
    (4, 103, 203, 303, 403, 'france')
""")

Check table content

In [ ]:
spark.sql(f"""
SELECT * FROM iceberg_catalog_region1.{DB_NAME}.{TABLE_NAME}
""").show()

## Create backup DB and table in another region

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS iceberg_catalog_region2.{DB_NAME} LOCATION '{S3_ROOT_REGION_2}/{DB_NAME}/'
""")

In [ ]:
#Get current time with format YYYYMMDDHHMMSS
from datetime import datetime
current_time = datetime.now().strftime("%Y%m%d%H%M%S")
current_time

In [ ]:
spark.sql(f"""
CREATE TABLE iceberg_catalog_region2.{DB_NAME}.{TABLE_NAME}_bkp_{current_time}
USING ICEBERG
PARTITIONED BY (country)
LOCATION '{S3_ROOT_REGION_2}/{DB_NAME}/{TABLE_NAME}_bkp_{current_time}/'
AS (SELECT * from iceberg_catalog_region1.{DB_NAME}.{TABLE_NAME})
""")

Validate backup table has been created

In [ ]:
spark.sql(f"""
SELECT * FROM iceberg_catalog_region2.{DB_NAME}.{TABLE_NAME}_bkp_{current_time}
""").show()